# Project 69 — Local Memory Strategy Benchmark

**Compare no-memory, buffer, and summary memory strategies for conversations.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (qwen3:8b) |
| Framework | LangChain |
| Interface | Jupyter |

## 1 · What You Will Learn

1. How **memory strategies** affect conversational quality
2. Implement **buffer memory** (keep last N messages)
3. Implement **summary memory** (compress history)
4. **Benchmark recall** across strategies

## 2 · Why This Project Matters

The wrong memory strategy causes forgotten details or context overflow.
This benchmark helps you choose the right approach.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports

In [ ]:
import json, time
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.3)
print(f'LLM ready: {MODEL}')

## 5 · Test Conversations

In [ ]:
CONVERSATIONS = [
    {'name': 'personal_info',
     'turns': ['My name is Alex and I work at DataFlow.', 'We have 12 employees.',
               'Our biggest customer is First National bank.',
               'What is my company called?', 'How many employees?'],
     'recall': {3: 'DataFlow', 4: '12'}},
    {'name': 'project_planning',
     'turns': ['Product launch March 15th.', 'Budget is $50,000, team of 5.',
               'Competitor is TechRival.', 'Features: real-time sync, offline mode, API.',
               'What is our launch date?', 'What features did I list?'],
     'recall': {4: 'March 15', 5: 'real-time'}},
]
print(f'{len(CONVERSATIONS)} test conversations')

## 6 · Memory Strategies

In [ ]:
class NoMemory:
    def add(self, role, content): self._last = content if role == 'user' else ''
    def get_context(self): return getattr(self, '_last', '')

class BufferMemory:
    def __init__(self, n=6): self.msgs, self.n = [], n
    def add(self, role, content):
        self.msgs.append(f'{role}: {content}')
        self.msgs = self.msgs[-self.n:]
    def get_context(self): return '\n'.join(self.msgs)

class SummaryMemory:
    def __init__(self): self.summary, self.recent = '', []
    def add(self, role, content):
        self.recent.append(f'{role}: {content}')
        if len(self.recent) > 4:
            old = '\n'.join(self.recent[:-2])
            p = ChatPromptTemplate.from_messages([('system','Summarize preserving all facts.'),('human','{text}')])
            self.summary = (p | llm | StrOutputParser()).invoke({'text': f'{self.summary}\n{old}'})
            self.recent = self.recent[-2:]
    def get_context(self):
        r = '\n'.join(self.recent)
        return f'Summary: {self.summary}\nRecent: {r}' if self.summary else r

print('Strategies: NoMemory, BufferMemory, SummaryMemory')

## 7 · Run Benchmark

In [ ]:
STRATEGIES = {'no_memory': NoMemory, 'buffer_6': lambda: BufferMemory(6), 'summary': SummaryMemory}

results = []
for conv in CONVERSATIONS:
    for sname, make_mem in STRATEGIES.items():
        mem = make_mem()
        for i, turn in enumerate(conv['turns']):
            mem.add('user', turn)
            ctx = mem.get_context()
            prompt = ChatPromptTemplate.from_messages([
                ('system', 'Use context to answer.'), ('human', 'Context:\n{ctx}\nMessage: {msg}')])
            t0 = time.time()
            resp = (prompt | llm | StrOutputParser()).invoke({'ctx': ctx, 'msg': turn})
            lat = time.time() - t0
            mem.add('assistant', resp)
            if i in conv.get('recall', {}):
                expected = conv['recall'][i]
                recalled = expected.lower() in resp.lower()
                results.append({'conv': conv['name'], 'strategy': sname, 'turn': i,
                                'expected': expected, 'recalled': recalled, 'latency': round(lat,2)})
                print(f'  {sname:12s} | {conv["name"]:15s} | [{"OK" if recalled else "MISS"}] {expected}')
print(f'\n{len(results)} recall tests')

## 8 · Results

In [ ]:
df = pd.DataFrame(results)
print('RECALL BY STRATEGY:')
print(df.groupby('strategy')['recalled'].mean().round(3).to_string())

## 9 · Save

In [ ]:
df.to_csv('memory_benchmark.csv', index=False)
print('Saved.')

## 10 · Limitations & Key Takeaways

**Limitations:** Short conversations, keyword-only recall test, no vector memory

**Takeaways:**
- Buffer memory provides reliable recall for recent context
- Summary memory trades exact recall for longer conversation support
- No memory fails at basic recall — always use some memory strategy